# Exponential and ordinary series via duality

`series.py` (in `algcom/`) defines two graded algebras directly, by giving
the product rule on the basis $t^n$:

- `OrdinarySeries`: $t^n \cdot t^m = t^{n+m}$ (plain convolution).
- `ExponentialSeries`: $t^n \cdot t^m = \binom{n+m}{n}\, t^{n+m}$ (Cauchy convolution).

This notebook builds the *same* two products the other way round, following
the duality construction of `20xHopfAlgebras.tex`, Sec. "Dualising a Hopf
algebra": start from the polynomial ring $Q[t]$ with basis $t^n$, define a
coproduct $\Delta$ on it, and dualise -- i.e. turn functionals on $Q[t]$
into an algebra via $f\star g := m_Q \circ (f\otimes g)\circ\Delta$.
This is exactly the classical convolution of distributions from
`main.tex`'s "Algebraic Probability" section,
$(F_1 * F_2)(x^n) := (F_1\otimes F_2)\circ\Delta(x^n)$.

At the end we check numerically that the two routes -- direct product rule,
and product dualised from a coproduct -- give the same algebra.

In [ ]:
import sys
from pathlib import Path

repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "algcom").exists():
        repo_root = candidate
        break
if repo_root is not None:
    sys.path.insert(0, str(repo_root))

from math import comb, factorial
from fractions import Fraction

from algcom import SparseVector, Algebra
from algcom.polynomial import Polynomial
from algcom.series import OrdinarySeries, ExponentialSeries

## 1. Choose the basis, define the coproducts

$\newcommand{\shuffle}{\mathtt{sh}}$

Basis: $t^n$, $n \ge 0$ (same basis `Polynomial` already uses). Two
coproducts on this basis, both landing in $Q[t]\otimes Q[t]$, represented
as a `SparseVector` whose keys are the tensor pairs $(a,b)$:

- **Deconcatenation** $\Delta_\cdot(t^n) := \sum_{a+b=n} t^a\otimes t^b$
  -- the coproduct dual to *plain* multiplication (Sec. "Dualising a Hopf
  algebra", `20xHopfAlgebras.tex`).
- **$\Delta_\shuffle$** $(t^n) := \sum_{k=0}^n \binom{n}{k}\, t^{n-k}\otimes t^k$
  -- the "unshuffle" coproduct of `20xHopfAlgebras.tex`, Example after
  Def. `eq.unshuffle.coalgebra`, and of `mathdef.tex`'s classical convolution.

Both are defined on a single basis element $t^n$ first, then lifted linearly
to any polynomial.

In [2]:
def deconcatenation_basis(n):
    """Delta_dot(t^n) = sum_{a+b=n} t^a (x) t^b."""
    result = SparseVector()
    for a in range(n + 1):
        result[(a, n - a)] += 1
    return result


def delta_shuffle_basis(n):
    """Delta_shuffle(t^n) = sum_k C(n,k) t^{n-k} (x) t^k."""
    result = SparseVector()
    for k in range(n + 1):
        result[(n - k, k)] += comb(n, k)
    return result


def lift_coproduct(basis_rule):
    """Extend a coproduct given on a single t^n to any polynomial, by
    linearity."""
    def lifted(p):
        result = SparseVector()
        for key, coeff in p._data.items():
            for ab_key, c in basis_rule(key.value)._data.items():
                result[ab_key.value] += coeff * c
        return result
    return lifted


Delta_dot = lift_coproduct(deconcatenation_basis)
Delta_shuffle = lift_coproduct(delta_shuffle_basis)

print("Delta_dot(t^2)     =", (deconcatenation_basis(2)))
print("Delta_shuffle(t^2) =", (delta_shuffle_basis(2)))
# by hand: Delta_shuffle(t^2) = t^0(x)t^2 + 2 t^1(x)t^1 + t^2(x)t^0

Delta_dot(t^2)     = ⟅(0, 2) + (1, 1) + (2, 0)⟆
Delta_shuffle(t^2) = ⟅(0, 2) + 2 (1, 1) + (2, 0)⟆


Sanity check against the worked example in `20xHopfAlgebras.tex`
(`eq.unshuffle.coalgebra`): $\Delta_\shuffle(t^2) = t^0\otimes t^2 + 2\, t^1\otimes t^1 + t^2\otimes t^0$,
matching the printout above. `Delta_dot` and `Delta_shuffle` also extend to
a genuine (multi-term) polynomial by linearity, as required:

In [3]:
p = Polynomial({1: 1, 2: 1})   # t + t^2
print("Delta_dot(t + t^2)     =", (Delta_dot(p)))
print("Delta_shuffle(t + t^2) =", (Delta_shuffle(p)))

Delta_dot(t + t^2)     = ⟅(0, 1) + (0, 2) + (1, 0) + (1, 1) + (2, 0)⟆
Delta_shuffle(t + t^2) = ⟅(0, 1) + (0, 2) + (1, 0) + 2 (1, 1) + (2, 0)⟆


## 2. Dualise: turn the coproduct into a product on functionals

A functional (a "distribution", in the language of `main.tex`) on $Q[t]$
is determined by its raw values $f_n := f(t^n)$ -- i.e. exactly the same
kind of object as an `OrdinarySeries`/`ExponentialSeries`: a `SparseVector`
keyed by degree. Dualising the coproduct means defining, for two such
functionals $f,g$,
$$
  \langle f\star g, t^n\rangle := \langle f\otimes g, \Delta(t^n)\rangle
      = \sum_{(a,b)} \Delta(t^n)[(a,b)]\; f_a\, g_b .
$$
Because $\Delta(t^n)$ is graded (every tensor pair $(a,b)$ it contains has
$a+b=n$), this pairing is determined pair-by-pair -- so it can be written
exactly in the "rule on basis elements" shape that `Algebra.from_rule`
expects, letting us reuse the same lifting machinery as the direct route
instead of writing a new bilinear-extension loop by hand.

In [4]:
def dual_rule_from_coproduct(basis_rule):
    """Turn a graded coproduct on basis elements into the basis-pair rule
    of its dual product: rule(p,q) picks out the (p,q) section coefficient
    of Delta(p+q)."""
    def rule(p, q):
        n = p + q
        coeff = basis_rule(n)[(p, q)]
        return SparseVector({n: coeff}) if coeff != 0 else SparseVector()
    return rule


one = SparseVector({0: 1})
dual_deconcat_algebra = Algebra.from_rule(dual_rule_from_coproduct(deconcatenation_basis), one=one)
dual_shuffle_algebra = Algebra.from_rule(dual_rule_from_coproduct(delta_shuffle_basis), one=one)

f = SparseVector({0: 1, 1: 2, 2: 3})
g = SparseVector({0: 1, 1: -1, 2: 5})

print("f star g   (dual of Delta_dot)     =", (dual_deconcat_algebra.m(f, g)))
print("f star g   (dual of Delta_shuffle) =", (dual_shuffle_algebra.m(f, g)))

f star g   (dual of Delta_dot)     = ⟅0 + 1 + 6 2 + 7 3 + 15 4⟆
f star g   (dual of Delta_shuffle) = ⟅0 + 1 + 4 2 + 21 3 + 90 4⟆


## 3. Check that the dual route and the direct route coincide

`series.py` defined `OrdinarySeries`/`ExponentialSeries` directly, from a
hand-written combinatorial rule. Here the *same* two products were derived
purely from a coproduct, via the general pairing formula. They should agree
exactly, on any input:

In [5]:
direct_ogf = OrdinarySeries.algebra().m(OrdinarySeries(f), OrdinarySeries(g))
dual_ogf = dual_deconcat_algebra.m(f, g)
check_ogf = "OK" if SparseVector(direct_ogf) == dual_ogf else "FAIL"
print(f"OrdinarySeries:    direct = {dict(direct_ogf)}")
print(f"                   dual   = {dict(dual_ogf)}   [{check_ogf}]")

direct_egf = ExponentialSeries.algebra().m(ExponentialSeries(f), ExponentialSeries(g))
dual_egf = dual_shuffle_algebra.m(f, g)
check_egf = "OK" if SparseVector(direct_egf) == dual_egf else "FAIL"
print(f"ExponentialSeries: direct = {dict(direct_egf)}")
print(f"                   dual   = {dict(dual_egf)}   [{check_egf}]")

assert SparseVector(direct_ogf) == dual_ogf
assert SparseVector(direct_egf) == dual_egf

OrdinarySeries:    direct = {BasisElement(0): Fraction(1, 1), BasisElement(1): Fraction(1, 1), BasisElement(2): Fraction(6, 1), BasisElement(3): Fraction(7, 1), BasisElement(4): Fraction(15, 1)}
                   dual   = {BasisElement(0): Fraction(1, 1), BasisElement(1): Fraction(1, 1), BasisElement(2): Fraction(6, 1), BasisElement(3): Fraction(7, 1), BasisElement(4): Fraction(15, 1)}   [OK]
ExponentialSeries: direct = {BasisElement(0): Fraction(1, 1), BasisElement(1): Fraction(1, 1), BasisElement(2): Fraction(4, 1), BasisElement(3): Fraction(21, 1), BasisElement(4): Fraction(90, 1)}
                   dual   = {BasisElement(0): Fraction(1, 1), BasisElement(1): Fraction(1, 1), BasisElement(2): Fraction(4, 1), BasisElement(3): Fraction(21, 1), BasisElement(4): Fraction(90, 1)}   [OK]


`Algebra.exponential`/`Algebra.logarithm` are generic (see `algebra.py`):
they only use `self.m` and `self.unit`, so they immediately work on
`dual_shuffle_algebra` too, with no extra code. Reusing the standard-normal
moments $m_n = \frac{n!}{2^{n/2}(n/2)!}$ ($n$ even, else $0$) from
`main.tex`, the $\star$-logarithm should give the cumulant series
$K(t) = t^2/2!$ (variance $1$, all higher cumulants $0$) -- exactly what
`ExponentialSeries.cumulants` (the direct route) gives.

In [6]:
order = 8

def normal_moment(n):
    if n % 2 == 1:
        return 0
    return Fraction(factorial(n), 2 ** (n // 2) * factorial(n // 2))

moments = {n: normal_moment(n) for n in range(order + 1)}

M_dual = SparseVector(moments)
K_dual = dual_shuffle_algebra.logarithm(M_dual, order=order)
K_dual = SparseVector({k.value: c for k, c in K_dual._data.items() if k.value <= order})

M_direct = ExponentialSeries.from_moments(moments)
K_direct = M_direct.cumulants(degree=order)

check_cum = "OK" if K_dual == SparseVector(K_direct) else "FAIL"
print(f"Cumulants, dual route   K = {dict(K_dual)}")
print(f"Cumulants, direct route K = {dict(K_direct)}   [{check_cum}]")

assert K_dual == SparseVector(K_direct)
print("\nDual and direct constructions agree: same algebra, two routes.")

Cumulants, dual route   K = {BasisElement(2): Fraction(1, 1)}
Cumulants, direct route K = {BasisElement(2): Fraction(1, 1)}   [OK]

Dual and direct constructions agree: same algebra, two routes.
